In [0]:
%run "../common/define_path"

In [0]:
tracking_raw_path
tracking_br_path

In [0]:
def ingest_raw_to_bronze( 
    source_path: str, 
    target_table: str, 
    schema_path: str, 
    checkpoint_path: str, 
    file_format: str = "csv" 
): 
    from pyspark.sql.functions import current_timestamp
    df = ( 
        spark.readStream 
            .format("cloudFiles") 
            .option("cloudFiles.format", file_format) 
            .option("cloudFiles.schemaLocation", schema_path) 
            .option("cloudFiles.inferColumnTypes", "true") 
            .option('header','true')
            .load(source_path) 
            .withColumn("ingestion_time", current_timestamp()) 
            
    ) 
 
    ( 
        df.writeStream 
          .format("delta") 
          .option("checkpointLocation", checkpoint_path) 
          .trigger(availableNow=True) 
          .table(target_table) 
    ) 

In [0]:
tables = [ 
  { 
    "source": f"{tracking_raw_path}/invoice/", 
    "target": "dev.tracking_br.invoice", 
    "schema": f"{tracking_br_path}/schema", 
    "checkpoint": f"{tracking_br_path}/checkpoint/invoice/",
    "format": "csv" 
  }, 
  { 
    "source": f"{tracking_raw_path}/order/", 
    "target": "dev.tracking_br.order", 
    "schema": f"{tracking_br_path}/schema", 
    "checkpoint": f"{tracking_br_path}/checkpoint/order/",
    "format": "csv" 
  },
  { 
    "source": f"{tracking_raw_path}/paymentdetail/", 
    "target": "dev.tracking_br.paymentdetail", 
    "schema": f"{tracking_br_path}/schema", 
    "checkpoint": f"{tracking_br_path}/checkpoint/paymentdetail/",
    "format": "csv" 
  },
  { 
    "source": f"{tracking_raw_path}/ordertracking/", 
    "target": "dev.tracking_br.ordertracking", 
    "schema": f"{tracking_br_path}/schema", 
    "checkpoint": f"{tracking_br_path}/checkpoint/ordertracking/",
    "format": "csv" 
  }   
]

for t in tables: 
    ingest_raw_to_bronze( 
        source_path=t["source"], 
        target_table=t["target"], 
        schema_path=t["schema"],  
        checkpoint_path=t["checkpoint"],
        file_format=t["format"] 
    )